# 02 — Match Simbad Targets with LSSTCam Object Catalogue

For each target star (`simbad_id`, `ra_deg`, `dec_deg`) read from the input CSV,
this notebook:

1. Locates the Butler tract/patch that contains the target coordinates.
2. Loads the `objectTable` for that **patch** (memory-efficient, cached).
3. Performs a nearest-neighbour sky cross-match with
   `astropy.coordinates.SkyCoord.match_to_catalog_sky`.
4. Retains matches within a configurable search radius.
5. Enriches each matched target with **all requested Rubin object columns**
   (`OBJ_INFO_COLUMNS` + per-band `OBJ_PERBAND_COLUMNS`) plus tract/patch flags.
6. Saves the merged table to CSV and Parquet (full + matched-only subsets).
7. Produces diagnostic plots: separation histogram, RA/Dec offsets,
   match-status pie chart, **colour-colour diagram (g−r vs r−i)**,
   and PSF r-band magnitude histogram.

---
**Reference:** https://github.com/lsst/tutorial-notebooks/blob/main/DP1/200_Data_Products/201_Catalogs/201_1_Object_table.ipynb

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-26 : found the proper way to load objects by tract patch in `object_patch` dataset
- **Last update:** 2026-06-27 : make it really works


## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patheffects import withStroke

from astropy.coordinates import SkyCoord
import astropy.units as u

from lsst.daf.butler import Butler
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

## 2. Logging

In [ ]:
# Configure a root logger that writes to stdout (captured by Jupyter).
log = logging.getLogger()
log.setLevel(logging.INFO)

# Avoid duplicate handlers when the cell is re-executed.
if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging configured.")

## 3. Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
# ── Notebook tag and I/O directories ─────────────────────────────────────────
NB_TAG = "MATCHOBJ_02"
DIR_DATA_IN = "data_DEEPCCUTOUTS_01_in"  # same input directory as NB 01
DIR_DATA_OUT = f"data_{NB_TAG}_out"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
log.info(f"Data input : {os.path.abspath(DIR_DATA_IN)}")
log.info(f"Data output: {os.path.abspath(DIR_DATA_OUT)}")
log.info(f"Figs       : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to both PDF and PNG inside DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Directory configuration done.")

In [ ]:
# ── Butler repository and collections ────────────────────────────────────────
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

# ── Input target file ─────────────────────────────────────────────────────────
target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"

# ── Cross-match search radius ─────────────────────────────────────────────────
MATCH_RADIUS_ARCSEC = 1.2  # maximum separation for a valid match [arcsec]

# ── Photometric bands ─────────────────────────────────────────────────────────
BANDS = "ugrizy"  # all bands to include in the output columns
BANDSEL = "r"  # reference band (used in single-band plots)
SURVEY_PREFIX = "lsst-obj"


# ── Object-table columns to retrieve ─────────────────────────────────────────
# Positional / global columns (band-independent)
OBJ_INFO_COLUMNS = [
    "tract",
    "patch",
    "coord_ra",
    "coord_raErr",
    "coord_dec",
    "coord_decErr",
    "coord_flag",
    "ebv",
]

# Per-band columns (will be prefixed with band letter, e.g. "r_psfFlux")
OBJ_PERBAND_FLUX_COLUMNS = [
    # aperture fluxes
    "ap12Flux",
    "ap12FluxErr",
    "ap17Flux",
    "ap17FluxErr",
    "ap25Flux",
    "ap25FluxErr",
    "ap50Flux",
    "ap50FluxErr",
    # PSF flux
    "psfFlux",
    "psfFluxErr",
]

OBJ_PERBAND_OTHER_COLUMNS = [
    # band-level coordinates
    "ra",
    "raErr",
    "dec",
    "decErr",
    # morphology / quality flags
    "sizeExtendedness",
    "sizeExtendedness_flag",
    "blendedness",
    "invalidPsfFlag",
    "inputCount",
]


# Expand per-band columns for every band
OBJ_BANDS_FLUX_COLUMNS = [f"{band}_{col}" for band in BANDS for col in OBJ_PERBAND_FLUX_COLUMNS]
OBJ_BANDS_OTHER_COLUMNS = [f"{band}_{col}" for band in BANDS for col in OBJ_PERBAND_OTHER_COLUMNS]

# Full list of object columns to attach to each matched target
OBJ_COLUMNS = OBJ_INFO_COLUMNS + OBJ_BANDS_FLUX_COLUMNS + OBJ_BANDS_OTHER_COLUMNS

log.info(f"OBJ_COLUMNS has {len(OBJ_COLUMNS)} entries.")
log.info(f"Selected OBJ columns {OBJ_COLUMNS}")
log.info("Butler configuration done.")

## 4. Helper utilities

In [ ]:
def dataset_type_exists(butler, name: str) -> bool:
    """Return True if *name* is a registered dataset type in *butler*."""
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False


def find_col(df: "pd.DataFrame", candidates: list) -> str:
    """Return the first column name from *candidates* present in *df*.

    Raises KeyError if none of the candidates are found.
    """
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found in table columns: {list(df.columns[:20])}")


def to_pandas(obj) -> "pd.DataFrame":
    """Convert any Butler table-like object to a pandas DataFrame."""
    if isinstance(obj, pd.DataFrame):
        return obj
    try:
        return obj.to_pandas()
    except AttributeError:
        return pd.DataFrame(obj)


def flux_to_ab_mag(flux, flux_err=None):
    """Convert a Rubin nJy PSF flux to AB magnitude.

    Parameters
    ----------
    flux : array-like
        Flux in nano-Jansky (nJy).
    flux_err : array-like, optional
        Flux uncertainty in nJy.

    Returns
    -------
    mag : ndarray
        AB magnitude (NaN for non-positive fluxes).
    mag_err : ndarray or None
        Magnitude uncertainty (only if flux_err supplied).
    """
    flux = np.asarray(flux, dtype=float)
    valid = flux > 0
    mag = np.full_like(flux, np.nan)
    mag[valid] = -2.5 * np.log10(flux[valid]) + 31.4  # nJy zero-point
    if flux_err is None:
        return mag
    flux_err = np.asarray(flux_err, dtype=float)
    mag_err = np.full_like(flux, np.nan)
    mag_err[valid] = 2.5 / np.log(10) * np.abs(flux_err[valid] / flux[valid])
    return mag, mag_err

## 5. Load Simbad targets

In [ ]:
df_targets = pd.read_csv(os.path.join(DIR_DATA_IN, target_file))
log.info(f"Loaded {len(df_targets)} targets from '{target_file}'")
display(df_targets.head())

## 6. Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
log.info(f"Butler initialised — repo: {repo}")

## 7. Auto-discover the object-table dataset type

The catalogue dataset name changed across pipeline versions:

| Pipeline era | Dataset type name     |
|:-------------|:----------------------|
| Gen2 / HSC   | `deepCoadd_obj`       |
| DP1          | `objectTable_tract`   |
| DP2+         | `object_patch`        |

We probe the registry so the notebook remains collection-agnostic.
See `00_QueryButler.ipynb` for an inventory of all available dataset types.


In [ ]:
# Prioritised candidate names (most specific first)
OBJECT_TABLE_CANDIDATES = [
    "object_patch",
    "objectTable",
    "object",
    "objectTable_tract",
    "object_table_tract",
    "deepCoadd_obj",
    "sourceTable_visit",
]

# Log all object/table dataset types registered in this collection
all_obj_types = [
    d.name for d in registry.queryDatasetTypes() if "object" in d.name.lower() or "table" in d.name.lower()
]
log.info(f"All object/table dataset types in registry: {all_obj_types}")

# Select the first candidate that is actually registered
OBJ_DATASET = None
for name in OBJECT_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        OBJ_DATASET = name
        log.info(f"Selected object-table dataset type: '{OBJ_DATASET}'")
        break

if OBJ_DATASET is None:
    raise RuntimeError(
        "No recognised object-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_obj_types}"
    )

## 8. Inspect the object-table schema

Load a single patch to discover the actual column names and filter
`OBJ_COLUMNS` down to columns that really exist in this collection.


In [ ]:
# Use the first target to locate a representative patch
first_row = df_targets.iloc[0]
probe_point = SpherePoint(first_row["ra_deg"], first_row["dec_deg"], degrees)
probe_tract_info = skymap.findTract(probe_point)
probe_patch_info = probe_tract_info.findPatch(probe_point)
probe_tract_id = probe_tract_info.getId()
probe_patch_id = probe_patch_info.getSequentialIndex()

probe_dataId = {
    "tract": probe_tract_id,
    "patch": probe_patch_id,
    "skymap": skymapName,
}

log.info(f"Probing schema with tract {probe_tract_id} / patch {probe_patch_id}")
df_probe = to_pandas(butler.get(OBJ_DATASET, dataId=probe_dataId))
log.info(f"Probe table: {len(df_probe):,} rows x {len(df_probe.columns)} columns")

In [ ]:
# Identify RA, Dec, and objectId column names (vary between pipeline versions)
ra_col = find_col(df_probe, ["coord_ra", "ra", "RA", "ra_deg"])
dec_col = find_col(df_probe, ["coord_dec", "dec", "DEC", "dec_deg"])
id_col = find_col(df_probe, ["objectId", "object_id", "id", "sourceId"])
log.info(f"RA column  : {ra_col}")
log.info(f"Dec column : {dec_col}")
log.info(f"ID column  : {id_col}")

# Filter OBJ_COLUMNS to only those actually present in this collection
available_cols = set(df_probe.columns)
OBJ_COLS_AVAIL = [c for c in OBJ_COLUMNS if c in available_cols]
OBJ_COLS_MISSING = [c for c in OBJ_COLUMNS if c not in available_cols]

log.info(
    f"OBJ_COLUMNS requested: {len(OBJ_COLUMNS)} | "
    f"available: {len(OBJ_COLS_AVAIL)} | "
    f"missing: {len(OBJ_COLS_MISSING)}"
)
if OBJ_COLS_MISSING:
    log.warning(f"Missing OBJ columns (will be skipped): {OBJ_COLS_MISSING}")

## 9. Patch-level object-table cache

Each Butler `get()` call is expensive. We cache the per-patch DataFrames so
that a patch containing multiple targets is loaded only once.


In [ ]:
# Cache keyed by (tract_id, patch_id)
patch_cache: dict = {}


def get_object_table_patch(tract_id: int, patch_id: int) -> "pd.DataFrame | None":
    """Return the cached (or freshly loaded) object table for one patch.

    Loading patch-by-patch instead of the full tract prevents RSP OOM crashes
    (a typical DP2 tract contains ~1 M objects; one patch ~5-20 k).
    """
    key = (tract_id, patch_id)
    if key in patch_cache:
        return patch_cache[key]

    dataId = {"tract": tract_id, "patch": patch_id, "skymap": skymapName}
    log.info(f"Loading object table: tract={tract_id}, patch={patch_id} ...")

    try:
        df = to_pandas(butler.get(OBJ_DATASET, dataId=dataId))
        log.info(f"  -> {len(df):,} objects loaded")
        patch_cache[key] = df
        return df
    except Exception as exc:
        log.warning(f"  -> FAILED (tract={tract_id}, patch={patch_id}): {exc}")
        return None


log.info("Patch cache initialised.")

## 10. Main cross-match loop

For each Simbad target:
1. Locate the containing tract and patch.
2. Load the patch object table (cached).
3. Nearest-neighbour match with `SkyCoord.match_to_catalog_sky`.
4. Accept the match if separation <= `MATCH_RADIUS_ARCSEC`.
5. Attach all available `OBJ_COLS_AVAIL` columns from the matched Rubin object.


In [ ]:
results = []  # one dict per target

for idx, target in df_targets.iterrows():
    simbad_id = target["simbad_id"]
    ra_t = target["ra_deg"]
    dec_t = target["dec_deg"]

    # ── 1. Find tract and patch ───────────────────────────────────────────────
    point = SpherePoint(ra_t, dec_t, degrees)
    tract_info = skymap.findTract(point)
    tract_id = tract_info.getId()
    patch_info = tract_info.findPatch(point)
    patch_id = patch_info.sequential_index

    log.info(
        f"[{idx:3d}] {simbad_id:40s} " f"ra={ra_t:.5f} dec={dec_t:+.5f} " f"tract={tract_id} patch={patch_id}"
    )

    # ── 2. Load patch object table (cached) ───────────────────────────────────
    df_obj = get_object_table_patch(tract_id, patch_id)

    # Base row: all Simbad target columns + tract/patch location
    row = target.to_dict()
    row["lsst_tract"] = tract_id
    row["lsst_patch"] = patch_id

    if df_obj is None or len(df_obj) == 0:
        log.warning(f"  No object table for tract={tract_id} / patch={patch_id} -- skipping")
        row.update(
            {
                "lsst_objectId": None,
                "lsst_ra": None,
                "lsst_dec": None,
                "separation_arcsec": None,
                "match_status": "no_table",
            }
        )
        # Fill OBJ columns with NaN so the output schema is uniform
        for col in OBJ_COLS_AVAIL:
            row[f"rubin_{col}"] = np.nan
        results.append(row)
        continue

    # ── 3. Build SkyCoord catalogue ───────────────────────────────────────────
    ra_obj = df_obj[ra_col].values
    dec_obj = df_obj[dec_col].values

    # Auto-detect radians vs degrees
    unit_sky = u.rad if ra_obj.max() <= 2 * np.pi + 0.1 else u.deg

    cat_sky = SkyCoord(ra=ra_obj * unit_sky, dec=dec_obj * unit_sky)
    tgt_sky = SkyCoord(ra=ra_t * u.deg, dec=dec_t * u.deg)

    # ── 4. Nearest-neighbour match ────────────────────────────────────────────
    best_idx, sep2d, _ = tgt_sky.match_to_catalog_sky(cat_sky)
    sep_arcsec = sep2d.to(u.arcsec).value[0]

    matched_obj = df_obj.iloc[best_idx]

    if sep_arcsec <= MATCH_RADIUS_ARCSEC:
        # Convert coordinates to degrees if stored in radians
        lsst_ra = matched_obj[ra_col]
        lsst_dec = matched_obj[dec_col]
        if unit_sky == u.rad:
            lsst_ra = np.degrees(lsst_ra)
            lsst_dec = np.degrees(lsst_dec)

        row.update(
            {
                "lsst_objectId": int(matched_obj[id_col]),
                "lsst_ra": lsst_ra,
                "lsst_dec": lsst_dec,
                "separation_arcsec": sep_arcsec,
                "match_status": "matched",
            }
        )

        # ── 5. Attach all available Rubin-LSST object columns ──────────────────────
        for col in OBJ_COLS_AVAIL:
            row[f"{SURVEY_PREFIX}_{col}"] = matched_obj[col]

        log.info(f"  matched  objectId={matched_obj[id_col]}  " f"sep={sep_arcsec:.3f} arcsec")
    else:
        row.update(
            {
                "lsst_objectId": None,
                "lsst_ra": None,
                "lsst_dec": None,
                "separation_arcsec": sep_arcsec,
                "match_status": "no_match",
            }
        )
        for col in OBJ_COLS_AVAIL:
            row[f"{SURVEY_PREFIX}_{col}"] = np.nan
        log.info(f"  no match  (closest sep={sep_arcsec:.3f} arcsec " f"> {MATCH_RADIUS_ARCSEC} arcsec)")

    results.append(row)

log.info(f"Done: {len(results)} targets processed.")

## 11. Results summary

In [ ]:
df_results = pd.DataFrame(results)

n_total = len(df_results)
n_matched = (df_results["match_status"] == "matched").sum()
n_nomatch = (df_results["match_status"] == "no_match").sum()
n_notable = (df_results["match_status"] == "no_table").sum()

log.info(f"Total targets  : {n_total}")
log.info(f"  Matched      : {n_matched}  ({100 * n_matched / n_total:.1f} %)")
log.info(f"  No match     : {n_nomatch}  (sep > {MATCH_RADIUS_ARCSEC} arcsec)")
log.info(f"  No table     : {n_notable}  (Butler load error)")

# Show the columns available in the enriched table
print(f"\nOutput columns ({len(df_results.columns)}):")
print(list(df_results.columns))

display(df_results.head())

## 12. Compute AB magnitudes from PSF fluxes

Rubin fluxes are stored in nano-Jansky (nJy). We convert `psfFlux` to AB
magnitudes for the diagnostic colour-colour plot.


In [ ]:
df_matched = df_results[df_results["match_status"] == "matched"].copy()

# Compute PSF magnitudes for g, r, i bands (used in colour-colour plot)
for band in ["u", "g", "r", "i", "z", "y"]:
    flux_col = f"{SURVEY_PREFIX}_{band}_psfFlux"
    fluxerr_col = f"{SURVEY_PREFIX}_{band}_psfFluxErr"
    mag_col = f"{SURVEY_PREFIX}_{band}_Mag"
    merr_col = f"{SURVEY_PREFIX}_{band}_MagErr"

    if flux_col in df_matched.columns:
        mag, mag_err = flux_to_ab_mag(
            df_matched[flux_col].values,
            df_matched[fluxerr_col].values if fluxerr_col in df_matched.columns else None,
        )
        df_matched[mag_col] = mag
        df_matched[merr_col] = mag_err
        log.info(
            f"Band {band}: {(~np.isnan(mag)).sum()} valid magnitudes " f"(median = {np.nanmedian(mag):.2f})"
        )
    else:
        log.warning(f"Column '{flux_col}' not available -- skipping mag_{band}")
        df_matched[mag_col] = np.nan
        df_matched[merr_col] = np.nan

## 13. Diagnostic plots

In [ ]:
# ── 13.1  Cross-match separation histogram ────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(
    df_matched["separation_arcsec"],
    bins=20,
    edgecolor="k",
    linewidth=0.5,
    color="steelblue",
)
ax.axvline(
    MATCH_RADIUS_ARCSEC,
    color="red",
    linestyle="--",
    label=f'search radius = {MATCH_RADIUS_ARCSEC}"',
)
ax.set_xlabel("Separation (arcsec)")
ax.set_ylabel("Number of targets")
ax.set_title("Cross-match separation -- Simbad targets vs LSST objects")
ax.legend()
plt.tight_layout()
savefig("crossmatch_separation_histogram")
plt.show()

In [ ]:
# ── 13.2  RA / Dec positional offsets (Simbad - LSST) ────────────────────────
cos_dec = np.cos(np.radians(df_matched["dec_deg"].values))
d_ra = (df_matched["ra_deg"].values - df_matched["lsst_ra"].values) * cos_dec * 3600
d_dec = (df_matched["dec_deg"].values - df_matched["lsst_dec"].values) * 3600

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(d_ra, d_dec, s=20, alpha=0.7, color="steelblue")
circle = plt.Circle(
    (0, 0),
    MATCH_RADIUS_ARCSEC,
    color="red",
    fill=False,
    linestyle="--",
    label=f'r = {MATCH_RADIUS_ARCSEC}"',
)
ax.add_patch(circle)
ax.axhline(0, color="k", linewidth=0.5)
ax.axvline(0, color="k", linewidth=0.5)
ax.set_xlabel(r"$\Delta\,\alpha\cos\delta$ (arcsec)")
ax.set_ylabel(r"$\Delta\,\delta$ (arcsec)")
ax.set_title("Positional offsets: Simbad - LSST")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
savefig("crossmatch_radec_offsets")
plt.show()

In [ ]:
# ── 13.3  Match-status pie chart ──────────────────────────────────────────────
counts = df_results["match_status"].value_counts()
fig, ax = plt.subplots(figsize=(4, 4))
ax.pie(
    counts.values,
    labels=counts.index,
    autopct="%1.0f%%",
    startangle=90,
    colors=["#4CAF50", "#F44336", "#FF9800"],
)
ax.set_title(f'Match status (r < {MATCH_RADIUS_ARCSEC}")')
plt.tight_layout()
savefig("crossmatch_status_pie")
plt.show()

In [ ]:
# ── 13.4  PSF r-band magnitude histogram ──────────────────────────────────────
mag_r_col = f"{SURVEY_PREFIX}_r_Mag"
mag_r = df_matched[mag_r_col].dropna()

if len(mag_r) > 0:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(mag_r, bins=15, color="steelblue", edgecolor="k", linewidth=0.5)
    ax.set_xlabel("PSF AB magnitude (r-band)")
    ax.set_ylabel("Number of targets")
    ax.set_title("Distribution of r-band PSF magnitudes -- matched Simbad targets")
    plt.tight_layout()
    savefig("matched_r_mag_histogram")
    plt.show()
else:
    log.warning("No valid r-band magnitudes -- skipping PSF magnitude histogram.")

In [ ]:
# ── 13.5  Colour-colour diagram: (g-r) vs (r-i) ──────────────────────────────
# Select only objects with finite magnitudes in all three bands.
g_mag = f"{SURVEY_PREFIX}_g_Mag"
r_mag = f"{SURVEY_PREFIX}_r_Mag"
i_mag = f"{SURVEY_PREFIX}_i_Mag"


cc_mask = np.isfinite(df_matched[g_mag]) & np.isfinite(df_matched[r_mag]) & np.isfinite(df_matched[i_mag])

df_cc = df_matched[cc_mask].copy()

if len(df_cc) == 0:
    log.warning("No objects with valid g, r, i magnitudes -- skipping colour-colour plot.")
else:
    g_minus_r = df_cc[g_mag].values - df_cc[r_mag].values
    r_minus_i = df_cc[r_mag].values - df_cc[i_mag].values

    fig, ax = plt.subplots(figsize=(7, 6))

    sc = ax.scatter(
        r_minus_i,
        g_minus_r,
        c=df_cc[r_mag],
        cmap="viridis_r",
        s=50,
        alpha=0.85,
        edgecolors="k",
        linewidths=0.3,
        zorder=3,
    )
    cbar = fig.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label("PSF magnitude (r-band)")

    # Annotate each point with the Simbad identifier
    label_kw = dict(
        fontsize=6,
        ha="left",
        va="bottom",
        path_effects=[withStroke(linewidth=2, foreground="white")],
    )
    for i, row_cc in df_cc.iterrows():
        annotation = str(row_cc["simbad_id"]) + f" ({row_cc['mk_class_simple']})"
        ax.annotate(
            annotation,
            xy=(
                row_cc[r_mag] - row_cc[i_mag],
                row_cc[g_mag] - row_cc[r_mag],
            ),
            xytext=(3, 3),
            textcoords="offset points",
            **label_kw,
        )

    ax.set_xlabel(r"$r - i$ (AB mag)")
    ax.set_ylabel(r"$g - r$ (AB mag)")
    ax.set_title(f"{SURVEY_PREFIX} Colour-colour diagram: matched Simbad targets\n" "(PSF fluxes, Rubin DP2)")
    plt.tight_layout()
    savefig("color_color_g-r_vs_r-i")
    plt.show()
    log.info(f"Colour-colour plot: {len(df_cc)} objects with valid g, r, i magnitudes.")

## 14. Save enriched results

Two output files are written:

- **`targets_matched_lsst_objects`** — full results table (all targets,
  all statuses), including all available `rubin_*` columns for matched objects.
- **`targets_matched_only`** — matched-only subset for downstream notebooks.


In [ ]:
# ── Full results (all targets + Rubin columns) ────────────────────────────────
out_stem = os.path.join(DIR_DATA_OUT, "targets_matched_lsst_objects")
df_results.to_csv(out_stem + ".csv", index=False)
df_results.to_parquet(out_stem + ".parquet", index=False)
log.info(f"Full results saved: {out_stem}.{{csv,parquet}}  ({len(df_results)} rows)")

# ── Matched-only subset ───────────────────────────────────────────────────────
out_matched = os.path.join(DIR_DATA_OUT, "targets_matched_only")
df_matched.to_csv(out_matched + ".csv", index=False)
df_matched.to_parquet(out_matched + ".parquet", index=False)
log.info(f"Matched-only saved: {out_matched}.{{csv,parquet}}  ({n_matched} rows)")

log.info("\nOutput column groups:")
log.info(f"  Simbad input columns   : {list(df_targets.columns)}")
log.info(
    f"  Match metadata columns : lsst_tract, lsst_patch, lsst_objectId, lsst_ra, lsst_dec, separation_arcsec, match_status"
)
log.info(
    f"  Rubin-LSST object columns   : {[c for c in df_results.columns if c.startswith(f'{SURVEY_PREFIX}_')]}"
)